# TabDPT Classifier — DIMER end-to-end tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/tutorials/tabdpt_classifier_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Layer6%2FTabDPT-ffcc4d?style=flat)](https://huggingface.co/Layer6/TabDPT)
[![Upstream](https://img.shields.io/badge/Upstream-layer6ai--labs%2FTabDPT--inference-181717?style=flat&logo=github&logoColor=white)](https://github.com/layer6ai-labs/TabDPT-inference)
[![arXiv](https://img.shields.io/badge/arXiv-2608.01400-b31b1b.svg)](https://arxiv.org/abs/2608.01400)

**Profile:** `E2E`  
**Notebook specification:** DIMER Notebook Specification `1.0`  
**Repository code revision exercised:** `99eaf21986bd74a1336f3aa2ea32407eeea884ef`

This notebook demonstrates DIMER tabular **classification** with the repository's production-facing `TabDPTClassificationPipeline`. TabDPT is an in-context learner: `fit()` fits/records preprocessing state and registers a labelled support table for inference. **It does not perform gradient training or fine-tune the TabDPT weights.**

The upstream TabDPT v1.2 package and immutable model checkpoint supply the foundation model. This repository adds DIMER-facing model provenance and checksum verification, mixed-table preprocessing, schema enforcement, deterministic controls, evaluation helpers, and the `tabdpt-dimer-context-v3` serving artifact contract.

**By the end of this notebook you will be able to:**
- install an immutable repository revision with a pinned tutorial environment;
- resolve and SHA-256 verify the exact TabDPT v1.2 model weights;
- load a public sample or gated bring-your-own CSV data and validate its schema;
- condition TabDPT on labelled support data without gradient updates;
- evaluate classification with accuracy, log loss, ROC-AUC where defined, and a majority-class baseline;
- score genuinely separate new records and export machine-readable predictions;
- export the actual DIMER v3 serving artifact, reload it from serialized files, and verify prediction equivalence.

**This notebook does not demonstrate:** gradient fine-tuning, regression, causal inference, calibrated deployment thresholds, or production fitness. Tutorial metrics are demonstration evidence only. Public tabular data may overlap directly or indirectly with upstream pretraining, so these metrics are not clean benchmark evidence.


## Prerequisites and runtime contract

- **Environment:** fresh Google Colab or compatible Jupyter runtime; Python 3.11–3.13.
- **Accelerator:** GPU is recommended. CPU is supported but may be slower. This notebook forces `use_flash=False`, which is portable to Tesla T4 (`sm_75`) as well as newer GPUs.
- **Network:** required once to clone the pinned repository revision and acquire the pinned Hugging Face checkpoint unless already cached.
- **Data:** the default sample requires no private data. BYOD modes are optional and gated.
- **Privacy:** uploaded files stay in the notebook runtime unless you explicitly copy/export them elsewhere. Do not upload confidential, restricted, or sensitive data to a hosted notebook environment unless you are authorized to do so.
- **DIMER service controls:** default fitted-support ceiling `10,000` rows; default per-ensemble context `2,048`; allowed DIMER context range `128–16,384`. These are service controls, not intrinsic claims about the upstream model.

The setup cell intentionally refuses a non-fresh Python process where PyTorch is already imported, because the pinned install may replace core packages and would otherwise create a hidden restart boundary.


In [ ]:
import sys
if "torch" in sys.modules:
    raise RuntimeError(
        "Start from a fresh runtime: PyTorch is already imported, and the pinned tutorial install "
        "must complete before core ML packages are loaded."
    )

REPO_REVISION = "99eaf21986bd74a1336f3aa2ea32407eeea884ef"
REPO_DIR = "/content/tabdpt-classifier-pipeline"

!rm -rf "$REPO_DIR"
!git clone -q https://github.com/kurtvalcorza/tabdpt-classifier-pipeline.git "$REPO_DIR"
!git -C "$REPO_DIR" checkout -q "$REPO_REVISION"
!python -m pip install -q -r "$REPO_DIR/tutorials/requirements-colab.txt"
!python -m pip install -q --no-deps "$REPO_DIR"


## 1. Verify the runtime and model provenance

The next cell prints the effective runtime identity, then resolves the exact model checkpoint through repository code. The resolver is pinned to `Layer6/TabDPT` at an immutable Hugging Face revision and verifies the expected SHA-256 before model construction. No model-repository remote Python code is executed; the checkpoint is a `.safetensors` file and the inference implementation comes from the pinned `tabdpt==1.2.0` package.

A successful cell establishes **identity and byte integrity**, not model quality or deployment safety.


In [ ]:
import importlib.metadata as mdlib
import platform
import sys
import torch

from tabdpt_classifier_pipeline import (
    TABDPT_HF_REPO,
    TABDPT_HF_REVISION,
    TABDPT_PACKAGE_VERSION,
    TABDPT_UPSTREAM_CODE_COMMIT,
    TABDPT_WEIGHT_FILENAME,
    TABDPT_WEIGHT_SHA256,
    TabDPTClassificationPipeline,
)
from tabdpt_classifier_pipeline.pipeline import resolve_tabdpt_weights

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
for package in ["tabdpt", "torch", "numpy", "pandas", "scikit-learn", "huggingface-hub", "pyarrow"]:
    print(f"{package}:", mdlib.version(package))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
print("Tutorial attention mode: use_flash=False")

weights = resolve_tabdpt_weights()
print("Model repository:", TABDPT_HF_REPO)
print("Model revision:", TABDPT_HF_REVISION)
print("Upstream code commit:", TABDPT_UPSTREAM_CODE_COMMIT)
print("Weight file:", TABDPT_WEIGHT_FILENAME)
print("Expected/verified SHA-256:", TABDPT_WEIGHT_SHA256)
print("Resolved local path:", weights)


## 2. Load the default sample or bring your own data

Choose one mode below.

- `sample` — deterministic public `sklearn.datasets.load_breast_cancer` data. The notebook creates separate support, evaluation, and new-record partitions.
- `upload_single` — upload one CSV containing features plus the target; the notebook creates deterministic stratified support/evaluation/new-record partitions. Random splitting assumes rows are sufficiently independent.
- `upload_presplit` — upload `train.csv`, `val.csv`, and `new.csv`. Existing partitions are preserved. `train.csv` and `val.csv` require the target; `new.csv` must be unlabelled.

**Expected schema before upload:** one classification target column (default `target`) plus one or more feature columns. Feature names must be unique. The target must have at least two classes in support data; every evaluation class must occur in support data. `new.csv` must match the fitted feature schema, except configured drop columns may also be present.

For temporal, grouped, panel, patient, device, household, spatial, repeated-entity, or otherwise leakage-sensitive data, use `upload_presplit` with partitions created according to the domain boundary. Do not use random row splitting merely because it is convenient.


In [ ]:
DATA_MODE = "sample"  # @param ["sample", "upload_single", "upload_presplit"]
TARGET_COLUMN = "target"  # @param {type:"string"}
SEED = 42  # @param {type:"integer"}

import csv
from collections import Counter
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

DATA_DIR = Path("/content/tabdpt-tutorial-data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def read_csv_strict(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        try:
            header = next(reader)
        except StopIteration as exc:
            raise ValueError(f"{path.name} is empty") from exc
    duplicates = sorted(name for name, count in Counter(header).items() if count > 1)
    if duplicates:
        raise ValueError(f"{path.name} contains duplicate column names: {duplicates}")
    return pd.read_csv(path)

def validate_labeled(frame: pd.DataFrame, name: str) -> None:
    if TARGET_COLUMN not in frame.columns:
        raise ValueError(f"{name} is missing target column {TARGET_COLUMN!r}")
    if frame.columns.duplicated().any():
        raise ValueError(f"{name} contains duplicate column names")
    if frame[TARGET_COLUMN].isna().any():
        raise ValueError(f"{name} target contains missing values")
    if frame.drop(columns=[TARGET_COLUMN]).shape[1] == 0:
        raise ValueError(f"{name} must contain at least one feature column")

if DATA_MODE == "sample":
    frame = load_breast_cancer(as_frame=True).frame.copy()
    support_eval, new_labeled = train_test_split(
        frame, test_size=0.10, random_state=SEED, stratify=frame[TARGET_COLUMN]
    )
    support, evaluation = train_test_split(
        support_eval,
        test_size=2/9,
        random_state=SEED,
        stratify=support_eval[TARGET_COLUMN],
    )
    new_records = new_labeled.drop(columns=[TARGET_COLUMN]).copy()
    DATA_PROVENANCE = "scikit-learn breast cancer dataset; public tutorial sample"
elif DATA_MODE in {"upload_single", "upload_presplit"}:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Upload modes require Google Colab. In local Jupyter, place the expected CSV files "
            f"under {DATA_DIR} and adapt this acquisition cell only."
        ) from exc
    uploaded = files.upload()
    for name, payload in uploaded.items():
        (DATA_DIR / Path(name).name).write_bytes(payload)

    if DATA_MODE == "upload_single":
        csv_files = sorted(DATA_DIR.glob("*.csv"))
        if len(csv_files) != 1:
            raise ValueError("upload_single requires exactly one CSV file")
        frame = read_csv_strict(csv_files[0])
        validate_labeled(frame, csv_files[0].name)
        support_eval, new_labeled = train_test_split(
            frame, test_size=0.10, random_state=SEED, stratify=frame[TARGET_COLUMN]
        )
        support, evaluation = train_test_split(
            support_eval,
            test_size=2/9,
            random_state=SEED,
            stratify=support_eval[TARGET_COLUMN],
        )
        new_records = new_labeled.drop(columns=[TARGET_COLUMN]).copy()
        DATA_PROVENANCE = f"user upload: {csv_files[0].name}; deterministic stratified 70/20/10 split"
    else:
        expected = {name: DATA_DIR / name for name in ("train.csv", "val.csv", "new.csv")}
        missing = [name for name, path in expected.items() if not path.exists()]
        if missing:
            raise ValueError(f"upload_presplit is missing required files: {missing}")
        support = read_csv_strict(expected["train.csv"])
        evaluation = read_csv_strict(expected["val.csv"])
        new_records = read_csv_strict(expected["new.csv"])
        validate_labeled(support, "train.csv")
        validate_labeled(evaluation, "val.csv")
        if TARGET_COLUMN in new_records.columns:
            raise ValueError("new.csv must be unlabelled; remove the target column")
        DATA_PROVENANCE = "user-provided train.csv/val.csv/new.csv partitions preserved"
else:
    raise ValueError(f"Unsupported DATA_MODE: {DATA_MODE!r}")

validate_labeled(support, "support")
validate_labeled(evaluation, "evaluation")
support_classes = set(support[TARGET_COLUMN].map(str))
evaluation_classes = set(evaluation[TARGET_COLUMN].map(str))
missing_eval_classes = sorted(evaluation_classes - support_classes)
if missing_eval_classes:
    raise ValueError(f"Evaluation contains classes absent from support data: {missing_eval_classes}")

if len(support) > 10_000:
    raise ValueError(
        f"Support has {len(support):,} rows, exceeding the DIMER default support ceiling of 10,000. "
        "This tutorial refuses silent capping; provide an explicit, documented support subset."
    )

print("Data provenance:", DATA_PROVENANCE)
print("Support shape:", support.shape)
print("Evaluation shape:", evaluation.shape)
print("New-record shape:", new_records.shape)
print("Support classes:", sorted(support_classes))
print("Evaluation classes:", sorted(evaluation_classes))


## 3. Condition the model on support data

`TabDPTClassificationPipeline.fit()` is the repository's supported conditioning API. It fits the repository's mixed-table preprocessing state and registers the labelled support context with the upstream estimator. **No TabDPT weight receives a gradient update.**

Successful execution means that the support schema, labels, preprocessing, pinned model bytes, and in-context state were accepted. It does not establish generalization quality.


In [ ]:
pipe = TabDPTClassificationPipeline(
    model_weight_path=weights,
    compile_model=False,
    use_flash=False,
    seed=SEED,
)
pipe.fit(support, target_column=TARGET_COLUMN, seed=SEED)

print("Fitted feature count:", len(pipe.feature_encoder.feature_columns))
print("Class order:", pipe.class_labels_)
print("Adaptation semantics: preprocessing fit + in-context support conditioning; no gradient training")


## 4. Evaluate and compare a trivial baseline

The repository reports:

- **accuracy** — fraction of records assigned the correct discrete class;
- **log loss** — penalizes probability-like class scores assigned to the wrong/true classes and is sensitive to confidence;
- **ROC-AUC** — for binary classification only, measures ranking discrimination across thresholds.

These are **single-holdout tutorial metrics**. No dispersion estimate is computed, so do not treat the displayed values as stable across datasets or runs. The class scores returned by `predict_proba()` are not claimed to be calibrated probabilities; deployment calibration and threshold selection belong to the downstream application.

The default decision rule is **argmax over the class-score vector**. A majority-class accuracy baseline is included so the model is not evaluated against zero context.


In [ ]:
import json
from sklearn.metrics import accuracy_score

INFERENCE = {
    "n_ensembles": 2,
    "context_size": 512,
    "batch_size": 512,
    "temperature": 1.0,
    "seed": SEED,
}
if not (128 <= INFERENCE["context_size"] <= 16_384):
    raise ValueError("context_size is outside the DIMER supported range 128..16384")

metrics = pipe.evaluate(evaluation, **INFERENCE)
majority_class = support[TARGET_COLUMN].map(str).mode().iloc[0]
baseline_accuracy = float(
    accuracy_score(
        evaluation[TARGET_COLUMN].map(str),
        [majority_class] * len(evaluation),
    )
)

evaluation_report = {
    "estimationProcedure": "single deterministic holdout; no model selection uses this holdout",
    "tutorialEvidenceOnly": True,
    "modelMetrics": metrics,
    "majorityClassBaseline": {
        "class": majority_class,
        "accuracy": baseline_accuracy,
    },
}
print(json.dumps(evaluation_report, indent=2))


## 5. Run inference on separate new records

These records are separate from the evaluation partition. The output preserves a stable `row_id`, a discrete `prediction`, and one score column per class in the exact `pipe.class_labels_` order.

The scores come from the pipeline's `predict_proba()` interface. They support argmax classification and ranking, but this notebook does **not** establish calibration or a deployment-specific decision threshold.


In [ ]:
import numpy as np
import pandas as pd

new_scores = pipe.predict_proba(new_records, **INFERENCE)
new_predictions = pipe.predict(new_records, **INFERENCE)

prediction_table = pd.DataFrame({"row_id": new_records.index.astype(str), "prediction": new_predictions.astype(str)})
for class_name in pipe.class_labels_:
    prediction_table[f"score_{class_name}"] = new_scores[class_name].to_numpy()

print(prediction_table.head())
print("Score-column class order:", pipe.class_labels_)


## 6. Export machine-readable outputs, provenance, and the DIMER serving artifact

The DIMER deployable state is **not the checkpoint alone**. TabDPT serving also needs the labelled support context and the fitted preprocessing state. This notebook exports the current `tabdpt-dimer-context-v3` contract:

- `artifact.json` — task/model identity, class order, runtime configuration, and fitted preprocessing state;
- `training_context.parquet` — the exact labelled support table used to condition the model.

Because the artifact contains the support table, apply the same confidentiality, licensing, retention, and disclosure controls as the source data. The manifest contains no credentials.

The manifest's context SHA-256 proves internal byte consistency; it does **not** authenticate the sender. Model authenticity is separately anchored to the immutable Hugging Face revision and the repository-pinned model SHA-256.


In [ ]:
import hashlib
import json
from pathlib import Path
import importlib.metadata as mdlib

OUTPUT_DIR = Path("/content/tabdpt-tutorial-output")
ARTIFACT_DIR = OUTPUT_DIR / "artifact"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

predictions_path = OUTPUT_DIR / "predictions.csv"
metrics_path = OUTPUT_DIR / "metrics.json"
provenance_path = OUTPUT_DIR / "provenance.json"
context_path = ARTIFACT_DIR / "training_context.parquet"
manifest_path = ARTIFACT_DIR / "artifact.json"

prediction_table.to_csv(predictions_path, index=False)
metrics_path.write_text(json.dumps(evaluation_report, indent=2) + "\n", encoding="utf-8")
support.to_parquet(context_path, index=False)

preprocessing_state = pipe.export_preprocessing_state()
manifest = {
    "format": "tabdpt-dimer-context-v3",
    "taskType": "tabular_classification",
    "targetColumn": TARGET_COLUMN,
    "dropColumns": list(preprocessing_state["dropColumns"]),
    "classNames": list(pipe.class_labels_),
    "runtimeConfig": {"fine_tune": False, **INFERENCE},
    "preprocessing": preprocessing_state,
    "baseModel": {
        "repo": TABDPT_HF_REPO,
        "revision": TABDPT_HF_REVISION,
        "filename": TABDPT_WEIGHT_FILENAME,
        "sha256": TABDPT_WEIGHT_SHA256,
        "upstreamCodeCommit": TABDPT_UPSTREAM_CODE_COMMIT,
    },
    "trainingContext": {"path": context_path.name, "sha256": sha256_file(context_path)},
}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")

provenance = {
    "repository": "kurtvalcorza/tabdpt-classifier-pipeline",
    "repositoryRevision": REPO_REVISION,
    "notebookProfile": "E2E",
    "notebookSpecVersion": "1.0",
    "dataProvenance": DATA_PROVENANCE,
    "model": manifest["baseModel"],
    "classOrder": list(pipe.class_labels_),
    "inference": INFERENCE,
    "runtime": {
        package: mdlib.version(package)
        for package in ["tabdpt", "torch", "numpy", "pandas", "scikit-learn", "huggingface-hub", "pyarrow"]
    },
    "useFlash": False,
}
provenance_path.write_text(json.dumps(provenance, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("Predictions:", predictions_path)
print("Metrics:", metrics_path)
print("Provenance:", provenance_path)
print("Artifact manifest:", manifest_path)
print("Artifact context:", context_path)
print("Context SHA-256:", manifest["trainingContext"]["sha256"])


## 7. Verify the serialized artifact across a fresh reconstruction boundary

A successful in-memory object does not prove serialization works. The next cell records probe predictions, deletes the original pipeline object, copies only the serialized artifact files into a fresh directory, reconstructs a new `TabDPTClassificationPipeline` through `load_artifact()`, and compares outputs.

Discrete classes must match exactly. Floating-point class scores are compared with an explicit tolerance (`rtol=1e-6`, `atol=1e-7`). This demonstrates artifact reconstruction equivalence for the probe records; it is still not evidence of production fitness.


In [ ]:
import gc
import numpy as np
import shutil
from pathlib import Path

probe = new_records.iloc[: min(10, len(new_records))].copy()
before_classes = pipe.predict(probe, **INFERENCE).astype(str).to_numpy()
before_scores = pipe.predict_proba(probe, **INFERENCE).to_numpy()

del pipe
gc.collect()

RELOAD_DIR = Path("/content/tabdpt-artifact-reload")
if RELOAD_DIR.exists():
    shutil.rmtree(RELOAD_DIR)
RELOAD_DIR.mkdir(parents=True)
shutil.copy2(manifest_path, RELOAD_DIR / "artifact.json")
shutil.copy2(context_path, RELOAD_DIR / "training_context.parquet")

reloaded = TabDPTClassificationPipeline.load_artifact(
    RELOAD_DIR / "artifact.json",
    model_weight_path=weights,
    compile_model=False,
    use_flash=False,
    seed=SEED,
)
after_classes = reloaded.predict(probe, **INFERENCE).astype(str).to_numpy()
after_scores = reloaded.predict_proba(probe, **INFERENCE).to_numpy()

if not np.array_equal(before_classes, after_classes):
    raise AssertionError("Serialized artifact reload changed one or more discrete predictions")
if not np.allclose(before_scores, after_scores, rtol=1e-6, atol=1e-7):
    max_abs = float(np.max(np.abs(before_scores - after_scores)))
    raise AssertionError(f"Serialized artifact reload changed class scores; max abs diff={max_abs}")

print("Artifact reloaded from fresh directory: PASS")
print("Exact class-prediction equivalence: PASS")
print("Class-score equivalence within rtol=1e-6, atol=1e-7: PASS")


## Interpretation, limits, and next steps

A successful top-to-bottom run proves that this pinned tutorial environment can acquire and checksum-verify the specified TabDPT v1.2 weights, validate/condition on the demonstrated table, compute the repository's classification metrics, score separate new records, export machine-readable outputs and a DIMER v3 support-context artifact, and reconstruct equivalent predictions from serialized artifact files.

It **does not prove** that TabDPT is accurate, calibrated, fair, robust, secure, or suitable for a particular production domain. The public sample may overlap upstream pretraining. A single holdout has no uncertainty estimate. Random row splitting is inappropriate for leakage-sensitive data. Argmax is the default decision rule; application thresholds and calibration require domain-labelled evidence. The exported support context may contain sensitive source data and must be governed accordingly.

Before release, record a clean-runtime execution for the exact notebook/PR revision, including runtime, accelerator, outcome, and artifact reload result. Static notebook validation is not a substitute for that execution evidence.

**Useful next experiments:** repeat evaluation across domain-valid splits; assess calibration on labelled deployment-like data; test schema drift and unseen categories; compare context/ensemble settings under a fixed evaluation protocol; and run the companion `ARTIFACT-INFERENCE` notebook using the exported artifact plus genuinely external new input.
